# date_parser 수정본 재검증 (300장)

담당: 이수민

`date_parser`를 3차까지 수정한 뒤 새로 처음부터 짠 검증 노트북입니다.
OCR 설정은 서현이 `docs/ocr_fallback_experiment_results.md`에 적어둔 팀 baseline과
정확히 일치하도록 맞췄습니다 (MKLDNN 활성화, 문서방향분류 비활성화, EXIF+RGB+512px 사전 리사이즈).
`labels_300.csv` + `label_images` 폴더로 300장 전체를 실제 OCR + `date_parser`로 돌려서
정확도를 확인합니다.

이 노트북이 하는 일:
1. `labels_300.csv`를 읽어서 대상 이미지 300장 목록을 만든다
2. PaddleOCR로 실제 OCR을 돌린다 (팀 baseline 설정)
3. OCR 출력을 공통 형식으로 바꿔서 `date_parser.parse_expiration_date()`에 넣는다
4. `outputs/validation_recheck_predictions.csv`에 저장한다
5. `labels_300.csv`를 정답으로 삼아 정확도를 계산한다
6. 실패 원인을 4단계로 자동 분류 (OCR 미탐지 / OCR 인식 실패 / 파서 미추출 / 파서 선택·해석 오류)

## 1. 설정

본인 PC 경로에 맞게 필요하면 수정하세요.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "date_parser").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("date_parser 패키지를 찾을 수 없습니다. itda_OCR 폴더(또는 그 하위)에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ===== 본인 환경에 맞게 수정 =====
LABELS_CSV = Path("labels_300.csv")         # labels_300.csv 위치 (다른 곳에 두셨으면 여기만 수정)
IMAGE_DIR = Path("label_images")            # 300장 이미지 폴더 위치
WEIGHTS_DIR = Path("weights/paddleocr")     # PaddleOCR 가중치 폴더
OUTPUT_PATH = Path("outputs/validation_recheck_predictions.csv")
MAX_IMAGES = None                            # 테스트로 일부만 돌려볼 땐 숫자로, 전체는 None
# =================================

for p, label in [(LABELS_CSV, "라벨 CSV"), (IMAGE_DIR, "이미지 폴더"), (WEIGHTS_DIR, "가중치 폴더")]:
    if not p.exists():
        raise FileNotFoundError(f"{label}을(를) 찾을 수 없습니다: {p.resolve()}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("모든 경로 확인 완료")

## 2. date_parser가 최신본인지 확인 (선택, 안전장치)

3차 수정에서 추가된 심볼을 import 해봐서, 커널이 예전 코드를 캐시하고 있지 않은지 확인합니다.
에러가 나면 Kernel → Restart Kernel 하고 다시 실행하세요.

In [ ]:
from date_parser.extract import extract_date_tokens

assert extract_date_tokens("26,07.14"), "date_parser가 예전 버전입니다. extract.py를 최신으로 덮어쓰고 Kernel Restart 하세요."
print("date_parser 최신본 확인 완료")

## 3. 라벨 로드

In [ ]:
import pandas as pd

labels = pd.read_csv(LABELS_CSV, dtype=str, encoding="utf-8-sig").fillna("")
labels = labels.loc[:, ~labels.columns.str.startswith("Unnamed")]

required_cols = {"file_name", "image_id", "year", "month", "day", "final_date"}
missing_cols = required_cols - set(labels.columns)
if missing_cols:
    raise ValueError(f"labels_300.csv에 필요한 열이 없습니다: {missing_cols}")

if MAX_IMAGES is not None:
    labels = labels.iloc[:MAX_IMAGES].copy()

print(f"대상: {len(labels)}장")
labels.head()

## 4. PaddleOCR 초기화

서현이 `docs/ocr_fallback_experiment_results.md`에 적어둔 팀 baseline 설정을 그대로 씁니다:
MKLDNN 활성화, 문서 방향 분류/왜곡보정/텍스트줄 방향 분류는 모두 비활성화,
인식 batch 6, 검출 threshold 0.7.

In [ ]:
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

from paddleocr import PaddleOCR

# team baseline settings, matching what 서현 documented:
# MKLDNN enabled; doc orientation classify / unwarping / textline orientation all disabled
ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_detection_model_dir=str(WEIGHTS_DIR / "PP-OCRv5_mobile_det_infer"),
    text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
    text_recognition_model_dir=str(WEIGHTS_DIR / "korean_PP-OCRv5_mobile_rec_infer"),
    text_recognition_batch_size=6,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=True,
    cpu_threads=4,
)
print("PaddleOCR loaded (matching Seohyun's baseline settings)")

## 4-1. Image preprocessing (matching Seohyun's baseline)

Seohyun's documented pipeline: EXIF orientation fix -> convert to RGB -> resize longest side to 512px with LANCZOS (never upscale). We preprocess images ourselves instead of relying on PaddleOCR's internal resizing, so the input matches exactly.

In [ ]:
from PIL import Image, ImageOps
import numpy as np


def preprocess_like_seohyun(image_path: Path, max_side: int = 512):
    """EXIF orientation fix -> RGB -> resize longest side to max_side (LANCZOS, never upscale). Returns an RGB numpy array to feed straight into PaddleOCR predict()."""
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    w, h = img.size
    scale = max_side / max(w, h)
    if scale < 1:
        img = img.resize((round(w * scale), round(h * scale)), Image.LANCZOS)
    return np.array(img)

## 5. 공통 형식 변환 함수

In [ ]:
def paddleocr_to_common_format(res):
    """PaddleOCR predict() 결과 1건 -> [{"text","confidence","bbox"}, ...]"""
    return [
        {"text": text, "confidence": float(conf), "bbox": [[float(x), float(y)] for x, y in poly]}
        for poly, text, conf in zip(res["rec_polys"], res["rec_texts"], res["rec_scores"])
    ]

## 6. OCR + date_parser 실행

300장이라 시간이 좀 걸립니다 (장당 약 8~9초 → 대략 40~45분 예상). 20장마다 진행상황이 출력됩니다.

**중간에 멈춘 것처럼 보여도 Interrupt/정지 누르지 마세요** — 진행 상황이 초기화되어 처음부터 다시 돌아갑니다.
다른 프로그램을 동시에 돌리고 있다면 메모리 압박으로 느려진 것일 수 있으니 작업관리자로 CPU/메모리부터 확인하세요.

In [ ]:
from date_parser import parse_expiration_date
import time

rows = []
debug_rows = []  # detected_text 등, 실패 원인 진단용 (predictions.csv에는 안 들어감)
missing_images = []
start = time.time()

for position, (_, row) in enumerate(labels.iterrows(), start=1):
    image_path = IMAGE_DIR / row["file_name"]
    if not image_path.is_file():
        missing_images.append(row["file_name"])
        rows.append({"image_id": row["image_id"], "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"})
        debug_rows.append({"image_id": row["image_id"], "detected_text": "(이미지 파일 없음)", "box_count": 0})
        continue

    preprocessed = preprocess_like_seohyun(image_path, max_side=512)
    ocr_raw = ocr.predict(preprocessed, text_det_limit_side_len=512, text_det_limit_type="max", text_det_box_thresh=0.7)
    ocr_common = paddleocr_to_common_format(ocr_raw[0])
    result = parse_expiration_date(ocr_common)
    rows.append({"image_id": row["image_id"], **result})
    debug_rows.append({
        "image_id": row["image_id"],
        "detected_text": " | ".join(item["text"] for item in ocr_common),
        "box_count": len(ocr_common),
    })

    if position % 20 == 0 or position == len(labels):
        elapsed = time.time() - start
        print(f"  [{position}/{len(labels)}] 진행 중... ({elapsed:.1f}초, 장당 평균 {elapsed / position:.2f}초)")

elapsed = time.time() - start
print(f"\n완료: {len(labels)}장, 총 {elapsed:.1f}초 (장당 평균 {elapsed / len(labels):.2f}초)")
if missing_images:
    print(f"이미지 파일을 못 찾은 항목 {len(missing_images)}건: {missing_images[:10]}")

predictions = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
ocr_debug = pd.DataFrame(debug_rows, columns=["image_id", "detected_text", "box_count"])
predictions.head()

## 7. 저장

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH.resolve()}")

## 8. labels_300.csv 기준 정확도

앞뒤 공백 제거, `None`/`none`/`NONE`을 `NONE`으로 통일, 숫자만으로 된 값은 앞자리 0을 무시하고 비교합니다 (`04`와 `4`는 같음).

In [ ]:
import re

def normalize_component(value):
    value = str(value).strip()
    if value.casefold() == "none":
        return "NONE"
    if re.fullmatch(r"[0-9]+", value):
        return value.lstrip("0") or "0"
    return value

FIELDS = ["year", "month", "day"]

truth = labels[["image_id", *FIELDS]].copy()
truth["image_id"] = truth["image_id"].astype(str).str.strip()
for f in FIELDS:
    truth[f] = truth[f].map(normalize_component)

pred_eval = predictions[["image_id", *FIELDS]].copy()
pred_eval["image_id"] = pred_eval["image_id"].astype(str).str.strip()
for f in FIELDS:
    pred_eval[f] = pred_eval[f].map(normalize_component)

comparison = truth.merge(pred_eval, on="image_id", suffixes=("_true", "_pred"), how="left")
for f in FIELDS:
    comparison[f"{f}_correct"] = comparison[f"{f}_true"] == comparison[f"{f}_pred"]
comparison["final_date_correct"] = comparison[[f"{f}_correct" for f in FIELDS]].all(axis=1)

total = len(comparison)
print(f"총 {total}장")
for f in [*FIELDS, "final_date"]:
    correct = int(comparison[f"{f}_correct"].sum())
    print(f"{f} 정확도: {correct}/{total} = {correct / total:.2%}")

## 9. 실패 사례

In [ ]:
failures = comparison.loc[~comparison["final_date_correct"]].merge(
    labels[["image_id", "file_name", "final_date", "notes"]] if "notes" in labels.columns else labels[["image_id", "file_name", "final_date"]],
    on="image_id", how="left"
).merge(ocr_debug, on="image_id", how="left")
print(f"실패 {len(failures)}건")
failures[["image_id", "file_name", "year_true", "year_pred", "month_true", "month_pred",
          "day_true", "day_pred", "detected_text"]]

## 10. 실패 원인 자동 분류

- **OCR 미탐지**: 이미지에서 텍스트를 아예 못 찾음
- **OCR 인식 실패**: 텍스트는 찾았는데 정답 숫자가 그 안 어디에도 없음 (date_parser 책임 아님)
- **파서 미추출**: 정답 숫자는 OCR 텍스트 안에 있는데 date_parser가 날짜 형태로 인식 못함 (고칠 수 있음)
- **파서 선택/해석 오류**: 날짜는 추출했는데 최종 값이 틀림 (고칠 수 있음)

In [ ]:
import re
from date_parser.extract import extract_date_tokens


def normalize_digits(value):
    return re.sub(r"[^0-9]", "", str(value))


def expected_date_variants(year, month, day):
    try:
        year, month, day = int(year), int(month), int(day)
    except (TypeError, ValueError):
        return set()
    y4, y2 = f"{year:04d}", f"{year % 100:02d}"
    mm, dd = f"{month:02d}", f"{day:02d}"
    m, d = str(month), str(day)
    return {
        y4 + mm + dd, y2 + mm + dd, y4 + m + d, y2 + m + d,
        dd + mm + y4, dd + mm + y2, d + m + y4, d + m + y2,
        mm + dd + y4, mm + dd + y2, m + d + y4, m + d + y2,
    }


def date_recalled(text, year, month, day):
    variants = expected_date_variants(year, month, day)
    if not variants:
        return None
    digits = normalize_digits(text)
    return any(v in digits for v in variants)


def classify_failure(row):
    if row["box_count"] == 0:
        return "OCR 미탐지"
    recalled = date_recalled(row["detected_text"], row["year_true"], row["month_true"], row["day_true"])
    if recalled is None:
        return "정답에 NONE 포함(별도 검토)"
    if recalled is False:
        return "OCR 인식 실패"
    tokens = extract_date_tokens(row["detected_text"])
    if not tokens:
        return "파서 미추출"
    return "파서 선택/해석 오류"


failures["failure_category"] = failures.apply(classify_failure, axis=1)
failures["failure_category"].value_counts()

## 11. 고칠 수 있는 실패만 따로 보기 + 저장

In [ ]:
fixable = failures.loc[failures["failure_category"].isin(["파서 미추출", "파서 선택/해석 오류"])]
print(f"date_parser가 고칠 수 있는 실패: {len(fixable)}건")
display(fixable[["image_id", "file_name", "final_date", "year_pred", "month_pred", "day_pred",
         "detected_text", "failure_category"]])

failures.to_csv("outputs/recheck_failures_debug.csv", index=False, encoding="utf-8-sig")
print("저장 완료: outputs/recheck_failures_debug.csv")